In [33]:
import os
import pandas as pd
import numpy as np


# now dump the data to the pg database
import psycopg2
from psycopg2 import sql
from sqlalchemy import create_engine

In [34]:
file_name = 'extended/albion.csv'
print(f"Cargando datos... {file_name}")

# current_rival = get_rival_name(file_name)
# file = 'partidos/' + file_name
file = 'work/cargar_datos_base/partidos/' + file_name
# column_names = [
#     "Player Id", "Player Display Name", "Time", "Elapsed Time (s)",
#     "Lat", "Lon", "Speed km/h", "Instantaneous Acceleration Impulse", "Heart Rate (bpm)", ""
# ]


# Player Id, Player Display Name, Session Id, Time, Elapased Time, Speed (m/s), Instantaneous Acceleration Impulse, Heart Rate Interval, Latitude, Longitude,  Accl X, Accl Y, Accl Z, Gyro X, Gyro Y, Gyro Z, Hacc, Hdop, Quality of Signal, No. of Satellites
columnas_originales = [
    "Player Id", "Player Display Name", "Session Id", "Time", "Elapsed Time", "Speed (m/s)", "Instantaneous Acceleration Impulse",
    "Heart Rate Interval", "Latitude", "Longitude", "Accl X", "Accl Y", "Accl Z", "Gyro X", "Gyro Y", "Gyro Z", "Hacc", "Hdop", "Quality of Signal", "No. of Satellites"
]
datos_csv = pd.read_csv(file, delimiter=',', skiprows=1, encoding='latin1', names=columnas_originales)

Cargando datos... extended/albion.csv


In [35]:
datos_csv.head()

,Player Id,Player Display Name,Session Id,Time,Elapsed Time,Speed (m/s),Instantaneous Acceleration Impulse,Heart Rate Interval,Latitude,Longitude,Accl X,Accl Y,Accl Z,Gyro X,Gyro Y,Gyro Z,Hacc,Hdop,Quality of Signal,No. of Satellites
0,68,Del Pino Esquivo,1927,09:03:56.02,0.019998,1.13889,0.750001,0,-34.795304,-56.147873,0.1000,0.9375,0.4875,-102.4,102.4,-307.2,0,0.8,326,23
1,68,Del Pino Esquivo,1927,09:03:56.03,0.029998,1.13889,0.750001,0,-34.795304,-56.147873,0.0750,0.9250,0.4625,-102.4,102.4,-307.2,0,0.8,326,23
2,68,Del Pino Esquivo,1927,09:03:56.04,0.039998,1.13889,0.750001,0,-34.795304,-56.147873,0.0500,0.9125,0.4250,-102.4,102.4,-307.2,0,0.8,326,23
3,68,Del Pino Esquivo,1927,09:03:56.05,0.049998,1.13889,0.750001,0,-34.795304,-56.147873,0.0250,0.9000,0.4125,-102.4,51.2,-256.0,0,0.8,326,23
4,68,Del Pino Esquivo,1927,09:03:56.06,0.059998,1.13889,0.750001,0,-34.795304,-56.147873,0.0375,0.9000,0.4125,-102.4,0.0,-256.0,0,0.8,326,23


In [36]:
# now reformat the data, so it follows this schema
# column_names = ["Player Id", "Player Display Name", "Time", "Elapsed Time (s)","Lat", "Lon", "Speed km/h", "Instantaneous Acceleration Impulse", "Heart Rate (bpm)", ""]
# fix speed to km/h, and only keep two decimals

new_df = pd.DataFrame()
new_df["Player Id"] = datos_csv['Player Id']
new_df["Player Display Name"] = datos_csv['Player Display Name']
new_df["Time"] = datos_csv['Time']
new_df["Elapsed Time (s)"] = datos_csv['Elapsed Time']
new_df["Lat"] = datos_csv['Latitude']
new_df["Lon"] = datos_csv['Longitude']
new_df["Speed km/h"] = datos_csv['Speed (m/s)'] * 3.6
new_df["Speed km/h"] = new_df["Speed km/h"].apply(lambda x: round(x, 2))
new_df["Instantaneous Acceleration Impulse"] = datos_csv['Instantaneous Acceleration Impulse']
new_df["Heart Rate (bpm)"] = datos_csv['Heart Rate Interval']

new_df.head()



,Player Id,Player Display Name,Time,Elapsed Time (s),Lat,Lon,Speed km/h,Instantaneous Acceleration Impulse,Heart Rate (bpm)
0,68,Del Pino Esquivo,09:03:56.02,0.019998,-34.795304,-56.147873,4.1,0.750001,0
1,68,Del Pino Esquivo,09:03:56.03,0.029998,-34.795304,-56.147873,4.1,0.750001,0
2,68,Del Pino Esquivo,09:03:56.04,0.039998,-34.795304,-56.147873,4.1,0.750001,0
3,68,Del Pino Esquivo,09:03:56.05,0.049998,-34.795304,-56.147873,4.1,0.750001,0
4,68,Del Pino Esquivo,09:03:56.06,0.059998,-34.795304,-56.147873,4.1,0.750001,0


In [37]:
# top 10 max speed, check if the data is correct

top_10_max_speed = new_df.sort_values(by="Speed km/h", ascending=False).head(10)
top_10_max_speed

,Player Id,Player Display Name,Time,Elapsed Time (s),Lat,Lon,Speed km/h,Instantaneous Acceleration Impulse,Heart Rate (bpm)
6459304,245,Inti Lopez,10:45:16.55,6134.458244,-34.795003,-56.147785,34.01,5.388893,0
6459305,245,Inti Lopez,10:45:16.56,6134.468244,-34.795003,-56.147785,34.01,5.388893,0
6459306,245,Inti Lopez,10:45:16.57,6134.478244,-34.795003,-56.147785,34.01,5.388893,0
6459307,245,Inti Lopez,10:45:16.58,6134.488244,-34.795003,-56.147785,34.01,5.388893,0
6459308,245,Inti Lopez,10:45:16.59,6134.498244,-34.795003,-56.147785,34.01,5.388893,0
6459309,245,Inti Lopez,10:45:16.60,6134.508244,-34.795003,-56.147785,34.01,5.388893,0
6459310,245,Inti Lopez,10:45:16.61,6134.518244,-34.795003,-56.147785,34.01,5.388893,0
6459311,245,Inti Lopez,10:45:16.62,6134.528244,-34.795003,-56.147785,34.01,5.388893,0
6459312,245,Inti Lopez,10:45:16.63,6134.538244,-34.795003,-56.147785,34.01,5.388893,0
6459313,245,Inti Lopez,10:45:16.64,6134.548244,-34.795003,-56.147785,34.01,5.388893,0


In [38]:
# show max and min elapsed time by player
max_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].max()
min_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].min()
display(max_elapsed_time)
display(min_elapsed_time)

Player Display Name
Calione Fuentes        7256.998331
Cuadro Navarro         6891.099381
Del Pino Esquivo       7400.999419
EcheverrÃ­a Aznarez    6809.099016
Inti Lopez             7377.098008
Jhonatan Dalmas        7322.098997
Juan I Garcia          6840.998442
Nahuel Lopez           7270.098088
Tomas Viera            7143.996672
Vazquez Lautaro        8846.099213
Name: Elapsed Time (s), dtype: float64

Player Display Name
Calione Fuentes        0.020003
Cuadro Navarro         0.019996
Del Pino Esquivo       0.019998
EcheverrÃ­a Aznarez    0.019999
Inti Lopez             0.019998
Jhonatan Dalmas        0.019998
Juan I Garcia          0.020000
Nahuel Lopez           0.019998
Tomas Viera            0.019997
Vazquez Lautaro        0.019998
Name: Elapsed Time (s), dtype: float64

In [39]:
# now show the max and min times for each player
max_time = new_df.groupby("Player Display Name")["Time"].max()
min_time = new_df.groupby("Player Display Name")["Time"].min()
display(max_time)
display(min_time)


Player Display Name
Calione Fuentes        11:04:42.99
Cuadro Navarro         10:58:26.59
Del Pino Esquivo       11:07:16.99
EcheverrÃ­a Aznarez    10:57:45.49
Inti Lopez             11:05:59.19
Jhonatan Dalmas        11:06:10.19
Juan I Garcia          10:58:13.59
Nahuel Lopez           11:05:26.29
Tomas Viera            11:02:50.39
Vazquez Lautaro        11:04:13.89
Name: Time, dtype: object

Player Display Name
Calione Fuentes        09:03:46.02
Cuadro Navarro         09:03:35.52
Del Pino Esquivo       09:03:56.02
EcheverrÃ­a Aznarez    09:04:16.42
Inti Lopez             09:03:02.12
Jhonatan Dalmas        09:04:08.12
Juan I Garcia          09:04:12.62
Nahuel Lopez           09:04:16.22
Tomas Viera            09:03:46.42
Vazquez Lautaro        08:36:47.82
Name: Time, dtype: object

In [40]:
# # fix the elapsed times, there are player who have a minimum time near 0, which is ok, but there are other players who have a minimum time of 57000 seconds, which is not ok,
# # so we will fix this by substracting the minimum time to all the times of the player who have a minimum time greater than 1

# min_elapsed_time = new_df.groupby("Player Display Name")["Elapsed Time (s)"].min()
# players_to_fix = min_elapsed_time[min_elapsed_time > 1].index

# # Convert the "Time" column to datetime format
# new_df["Time"] = pd.to_datetime(new_df["Time"])

# for player in players_to_fix:
#     min_time = new_df[new_df["Player Display Name"] == player]["Time"].min()
#     new_df.loc[new_df["Player Display Name"] == player, "Time"] = new_df[new_df["Player Display Name"] == player]["Time"] - min_time
    



In [41]:
# sort data by player id and elapsed time
new_df = new_df.sort_values(by=['Player Id', 'Elapsed Time (s)'])

In [42]:
# save the data to a csv file
new_file_name = file.replace('extended/', 'cleaned/')

# persist the data to a csv file
new_df.to_csv(new_file_name, index=False)
